# Logistic Regression - Grid Search & Random Search

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import time
import itertools
import random
from data_loader import load_clinc

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = True
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

cuda
1525 / 620 / 1100, 151 classes (sample)


In [2]:
class TextDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class LogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

In [3]:
PATIENCE = 5
MAX_EPOCHS = 50


def evaluate_hyperparams(params):
    torch.manual_seed(42)
    np.random.seed(42)

    vectorizer = TfidfVectorizer(
        max_features=params['max_features'],
        ngram_range=(1, params['ngram_max']),
        min_df=2, max_df=0.95
    )
    tr_feat = vectorizer.fit_transform(train_texts).toarray()
    vl_feat = vectorizer.transform(val_texts).toarray()

    tr_loader = DataLoader(TextDataset(tr_feat, train_labels), batch_size=params['batch_size'], shuffle=True)
    vl_loader = DataLoader(TextDataset(vl_feat, val_labels), batch_size=params['batch_size'], shuffle=False)

    model = LogisticRegression(tr_feat.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])

    best_f1, patience_counter = 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for features, labels in tr_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds = []
        with torch.no_grad():
            for features, labels in vl_loader:
                all_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, all_preds, average='macro', zero_division=0)

        if vl_f1 > best_f1:
            best_f1, patience_counter = vl_f1, 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    return best_f1, epoch + 1

## Grid Search

In [4]:
grid = {
    'learning_rate': [0.001, 0.005, 0.01],
    'max_features': [3000, 5000, 8000],
    'ngram_max': [1, 2, 3],
    'batch_size': [64, 128],
    'weight_decay': [0, 1e-4]
}

keys = list(grid.keys())
combinations = list(itertools.product(*grid.values()))
print(f"Total combinations: {len(combinations)}")

grid_results = []
best_f1, best_params = 0, None
start_time = time.time()

for i, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    f1, epochs = evaluate_hyperparams(params)
    grid_results.append({"params": params, "val_f1": round(f1, 4), "epochs": epochs})

    if f1 > best_f1:
        best_f1, best_params = f1, params

    print(f"{i+1:3d}/{len(combinations)}  f1={f1:.4f}  {params}")

grid_time = time.time() - start_time
print(f"\nGrid Search done. Best F1: {best_f1:.4f}, Time: {grid_time:.1f}s")
print(f"Best params: {best_params}")

Total combinations: 108
  1/108  f1=0.6077  {'learning_rate': 0.001, 'max_features': 3000, 'ngram_max': 1, 'batch_size': 64, 'weight_decay': 0}
  2/108  f1=0.5536  {'learning_rate': 0.001, 'max_features': 3000, 'ngram_max': 1, 'batch_size': 64, 'weight_decay': 0.0001}
  3/108  f1=0.6054  {'learning_rate': 0.001, 'max_features': 3000, 'ngram_max': 1, 'batch_size': 128, 'weight_decay': 0}
  4/108  f1=0.5609  {'learning_rate': 0.001, 'max_features': 3000, 'ngram_max': 1, 'batch_size': 128, 'weight_decay': 0.0001}
  5/108  f1=0.6002  {'learning_rate': 0.001, 'max_features': 3000, 'ngram_max': 2, 'batch_size': 64, 'weight_decay': 0}
  6/108  f1=0.5639  {'learning_rate': 0.001, 'max_features': 3000, 'ngram_max': 2, 'batch_size': 64, 'weight_decay': 0.0001}
  7/108  f1=0.5794  {'learning_rate': 0.001, 'max_features': 3000, 'ngram_max': 2, 'batch_size': 128, 'weight_decay': 0}
  8/108  f1=0.5675  {'learning_rate': 0.001, 'max_features': 3000, 'ngram_max': 2, 'batch_size': 128, 'weight_decay': 

## Random Search

In [5]:
random_space = {
    'learning_rate': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05],
    'max_features': [1000, 2000, 3000, 5000, 8000, 10000],
    'ngram_max': [1, 2, 3],
    'batch_size': [32, 64, 128, 256],
    'weight_decay': [0, 1e-5, 1e-4, 1e-3]
}

N_RANDOM_ITER = len(combinations)  # same budget as grid search

random.seed(42)
random_results = []
best_f1_rand, best_params_rand = 0, None
start_time = time.time()

for i in range(N_RANDOM_ITER):
    params = {k: random.choice(v) for k, v in random_space.items()}
    f1, epochs = evaluate_hyperparams(params)
    random_results.append({"params": params, "val_f1": round(f1, 4), "epochs": epochs})

    if f1 > best_f1_rand:
        best_f1_rand, best_params_rand = f1, params

    print(f"{i+1:3d}/{N_RANDOM_ITER}  f1={f1:.4f}  {params}")

random_time = time.time() - start_time
print(f"\nRandom Search done. Best F1: {best_f1_rand:.4f}, Time: {random_time:.1f}s")
print(f"Best params: {best_params_rand}")

  1/108  f1=0.6538  {'learning_rate': 0.05, 'max_features': 1000, 'ngram_max': 1, 'batch_size': 128, 'weight_decay': 1e-05}
  2/108  f1=0.5492  {'learning_rate': 0.0005, 'max_features': 2000, 'ngram_max': 3, 'batch_size': 32, 'weight_decay': 0}
  3/108  f1=0.6393  {'learning_rate': 0.01, 'max_features': 5000, 'ngram_max': 1, 'batch_size': 32, 'weight_decay': 0}
  4/108  f1=0.5474  {'learning_rate': 0.0005, 'max_features': 2000, 'ngram_max': 3, 'batch_size': 32, 'weight_decay': 1e-05}
  5/108  f1=0.5985  {'learning_rate': 0.05, 'max_features': 10000, 'ngram_max': 3, 'batch_size': 256, 'weight_decay': 1e-05}
  6/108  f1=0.5947  {'learning_rate': 0.005, 'max_features': 8000, 'ngram_max': 2, 'batch_size': 32, 'weight_decay': 1e-05}
  7/108  f1=0.6528  {'learning_rate': 0.05, 'max_features': 5000, 'ngram_max': 2, 'batch_size': 128, 'weight_decay': 1e-05}
  8/108  f1=0.4344  {'learning_rate': 0.0005, 'max_features': 3000, 'ngram_max': 1, 'batch_size': 32, 'weight_decay': 0.001}
  9/108  f1=0

## Test Evaluation

In [6]:
def train_and_test(params, label):
    torch.manual_seed(42)
    np.random.seed(42)

    vectorizer = TfidfVectorizer(
        max_features=params['max_features'],
        ngram_range=(1, params['ngram_max']),
        min_df=2, max_df=0.95
    )
    tr_feat = vectorizer.fit_transform(train_texts).toarray()
    vl_feat = vectorizer.transform(val_texts).toarray()
    te_feat = vectorizer.transform(test_texts).toarray()

    tr_loader = DataLoader(TextDataset(tr_feat, train_labels), batch_size=params['batch_size'], shuffle=True)
    vl_loader = DataLoader(TextDataset(vl_feat, val_labels), batch_size=params['batch_size'], shuffle=False)
    te_loader = DataLoader(TextDataset(te_feat, test_labels), batch_size=params['batch_size'], shuffle=False)

    model = LogisticRegression(tr_feat.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])

    best_f1, best_state, patience_counter = 0, None, 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for features, labels in tr_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        vl_preds = []
        with torch.no_grad():
            for features, labels in vl_loader:
                vl_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, vl_preds, average='macro', zero_division=0)
        if vl_f1 > best_f1:
            best_f1, best_state, patience_counter = vl_f1, model.state_dict().copy(), 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    te_preds = []
    with torch.no_grad():
        for features, labels in te_loader:
            te_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

    acc = accuracy_score(test_labels, te_preds)
    p = precision_score(test_labels, te_preds, average='macro', zero_division=0)
    r = recall_score(test_labels, te_preds, average='macro', zero_division=0)
    f1_m = f1_score(test_labels, te_preds, average='macro', zero_division=0)
    f1_w = f1_score(test_labels, te_preds, average='weighted', zero_division=0)

    print(f"{label}:")
    print(f"    Accuracy:   {acc:.4f}")
    print(f"    Precision:  {p:.4f}")
    print(f"    Recall:     {r:.4f}")
    print(f"    F1 macro:   {f1_m:.4f}")
    print(f"    F1 weighted:{f1_w:.4f}")

    return {
        "accuracy": round(acc, 4), "macro_precision": round(p, 4),
        "macro_recall": round(r, 4), "macro_f1": round(f1_m, 4), "weighted_f1": round(f1_w, 4)
    }


grid_metrics = train_and_test(best_params, "Grid Search")
print()
rand_metrics = train_and_test(best_params_rand, "Random Search")

Grid Search:
    Accuracy:   0.6127
    Precision:  0.6308
    Recall:     0.6754
    F1 macro:   0.6270
    F1 weighted:0.5888

Random Search:
    Accuracy:   0.6164
    Precision:  0.6453
    Recall:     0.6862
    F1 macro:   0.6381
    F1 weighted:0.5947


## Save Results

In [7]:
gs_out = {
    "model_type": "LogisticRegression",
    "optimization": "grid_search",
    "best_hyperparameters": best_params,
    "total_evaluations": len(combinations),
    "search_time_seconds": round(grid_time, 2),
    "test_metrics": grid_metrics,
    "all_results": grid_results
}
with open('results/results_lr_grid.json', 'w') as f:
    json.dump(gs_out, f, indent=4)

rs_out = {
    "model_type": "LogisticRegression",
    "optimization": "random_search",
    "best_hyperparameters": best_params_rand,
    "total_evaluations": N_RANDOM_ITER,
    "search_time_seconds": round(random_time, 2),
    "test_metrics": rand_metrics,
    "all_results": random_results
}
with open('results/results_lr_random.json', 'w') as f:
    json.dump(rs_out, f, indent=4)

print("Saved: results/results_lr_grid.json, results/results_lr_random.json")

Saved: results/results_lr_grid.json, results/results_lr_random.json
